In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
df_train_cleaned=pd.read_csv("churn_train_cleaned.csv")
df_test=pd.read_csv("churn-bigml-20.csv")

In [33]:
x=df_train_cleaned.drop(columns=["Churn"])

In [34]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(df_train_cleaned["Churn"])

We used labelencoder here as the target variable is in bool value and ml understands only numbers so we encoded in that

In [35]:
y_df=pd.DataFrame(y)
y_df.value_counts()

0
0    2278
1     388
Name: count, dtype: int64

In [36]:
num_col=x.select_dtypes(include=["int64","float64"]).columns.tolist()

In [37]:
num_col

['Account length',
 'Area code',
 'International plan',
 'Voice mail plan',
 'Number vmail messages',
 'Total day minutes',
 'Total day calls',
 'Total eve minutes',
 'Total eve calls',
 'Total night minutes',
 'Total night calls',
 'Total intl minutes',
 'Total intl calls',
 'Customer service calls']

In [38]:
cat_col=x.select_dtypes(include=["object"]).columns.tolist()

In [39]:
cat_col

['State']

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [41]:
ct=ColumnTransformer([
                    ("num",StandardScaler(),num_col),
                    ("cat",OneHotEncoder(drop="first",sparse_output=False),cat_col)
    
])

We are gonna applying standard scaler in numerical column to standarize the value  and onehotencoder in categorial column 

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

In [43]:
baseline_pipeline=Pipeline(steps=[
    ("preprocessor",ct),
    ("classifier",LogisticRegression(max_iter=1000))
])

In [44]:
score=cross_validate(baseline_pipeline,x,y,cv=5,scoring=["accuracy","recall","precision","f1"])

In [45]:
print("baseline_accuracy: ",score["test_accuracy"].mean().round(4))
print("baseline_recall: ",score["test_recall"].mean().round(4))
print("baseline_precision",score["test_precision"].mean().round(4))
print("baseline_f1: ",score["test_f1"].mean().round(4))

baseline_accuracy:  0.8638
baseline_recall:  0.2397
baseline_precision 0.5837
baseline_f1:  0.3386


Here we can that using logistic regression we get around 86.3% accuracy but here precision is very around 58.3% that means only it could out of all churners it could identify only 58.3 were leaving and 42.7 percent are false positive. Also recall is 23.9 percent means model can identify actual true positives but has 76.1% is false negitive. And f1 score is really bad due to recall and precision

In [46]:
weighted_pipeline=Pipeline(steps=[
    ("preprocessor",ct),
    ("classifier",LogisticRegression(class_weight="balanced",max_iter=1000))
]
)

In [47]:
score_weighted=cross_validate(weighted_pipeline,x,y,cv=5,scoring=["accuracy","recall","precision","f1"])

In [48]:
print("weighted_accuracy: ",score_weighted["test_accuracy"].mean().round(4))
print("weighted_recall: ",score_weighted["test_recall"].mean().round(4))
print("weighted_precision",score_weighted["test_precision"].mean().round(4))
print("weighted_f1: ",score_weighted["test_f1"].mean().round(4))

weighted_accuracy:  0.7652
weighted_recall:  0.7293
weighted_precision 0.3534
weighted_f1:  0.4755


This model did pretty good as we can see that our accuracy is decreased but recall is significantly increased so this means model can highly predict true positive with minimal false negitive but this came with trade off of precison as it is decreased because the model was heavily penalised towrds catching the churners but it gets paranoid and start flagging the people who very little suspisious.  

In [49]:
from sklearn.ensemble import RandomForestClassifier

In [50]:
rf_pipeline=Pipeline(steps=[
    ("preprocessor",ct),
    ("classifier",RandomForestClassifier(class_weight="balanced",random_state=42))
])

In [51]:
score_rf=cross_validate(rf_pipeline,x,y,cv=5,scoring=["accuracy","recall","precision","f1"])

In [52]:
print("rf_accuracy: ",score_rf["test_accuracy"].mean().round(4))
print("rf_recall: ",score_rf["test_recall"].mean().round(4))
print("rf_precision",score_rf["test_precision"].mean().round(4))
print("rf_f1: ",score_rf["test_f1"].mean().round(4))

rf_accuracy:  0.9355
rf_recall:  0.7166
rf_precision 0.8198
rf_f1:  0.7636


Here we can see that we used random forest classifier because logistic regression is a linear model but a human behaviour isnt linear so we used used non linear model. We could have used decision tree but a single tree is prone to overfitting. Here accuracy is pretty good 93.5% and even precision jumped from 35% to 81.9%, recall stayed on same range. Out every 100 people model actucal can pridect 82 people as actual churner 18 are false positive and we got good f1 score too.

In [53]:
y_df.value_counts()

0
0    2278
1     388
Name: count, dtype: int64

Now we are gonna use xgboost for that here it doesnt use class weight it use scale pos weight which is false/true.
2278/388=5.87 approx is the scale_pos_weight parameter value

In [54]:
from xgboost import XGBClassifier

In [55]:
xgb_pipeline=Pipeline(steps=[
    ("preprocessor",ct),
    ("classifier",XGBClassifier(scale_pos_weight=5.87,random_state=42))
])

In [56]:
score_xgb=cross_validate(xgb_pipeline,x,y,cv=5,scoring=["accuracy","recall","precision","f1"])

In [57]:
print("xgb_accuracy: ",score_xgb["test_accuracy"].mean().round(4))
print("xgb_recall: ",score_xgb["test_recall"].mean().round(4))
print("xgb_precision",score_xgb["test_precision"].mean().round(4))
print("xgb_f1: ",score_xgb["test_f1"].mean().round(4))

xgb_accuracy:  0.9482
xgb_recall:  0.7784
xgb_precision 0.8534
xgb_f1:  0.8133
